# 06 · Simulate the 2026-27 standings

Finally we play the whole 2026-27 schedule **10,000 times**, sampling each game
from its probabilities and awarding Liiga points (regulation win 3, OT win 2, OT
loss 1, regulation loss 0). The result is a *distribution* of final tables.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

In [ ]:
from liiga.simulate import simulate
res = simulate()
standings = res['standings']
display(standings)

In [ ]:
import matplotlib.pyplot as plt
s = standings.iloc[::-1]   # best at top
err = [s.mean_points - s.p05_points, s.p95_points - s.mean_points]
plt.figure(figsize=(9,6))
plt.barh(s.team, s.mean_points, xerr=err, color='#3b7dd8')
plt.xlabel('points'); plt.title('Projected 2026-27 points (bar=mean, whiskers=5–95%)')
plt.tight_layout(); plt.show()

In [ ]:
# Position distribution heatmap — how likely is each team to finish in each spot?
import matplotlib.pyplot as plt
pos = res['position_distribution'].loc[standings['team']]
fig, ax = plt.subplots(figsize=(11,6))
im = ax.imshow(pos.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(pos.shape[1])); ax.set_xticklabels(pos.columns)
ax.set_yticks(range(len(pos))); ax.set_yticklabels(pos.index)
ax.set_xlabel('final position'); ax.set_title('P(team finishes in position)')
fig.colorbar(im, label='probability'); plt.tight_layout(); plt.show()

In [ ]:
print('Title and playoff odds:')
display(standings[['proj_rank','team','mean_points','p_title','p_top_playoff']])
print('\nJokerit (newly promoted):')
display(standings[standings.team=='Jokerit'])

## Tuning the forecast

Everything is driven by `config.yaml` and the editable CSVs in `data/`:

- **Rosters wrong?** Edit `data/rosters_2026_27.csv` → re-run **03 → 04 → 06**.
- **Trust team history more?** Raise `team_strength.team_weight` → re-run 04 → 06.
- **Home ice / OT lean / recency / regression** → edit `config.yaml` → re-run.

Switch `database.target` to `snowflake` to run the exact same pipeline in
production.